# Jittered A-B pairs

Each sound contains six pairs: low A, then high B. In one of two sounds, a late B arrives early. Which sound changed?

The pairs wander together by ±35 ms. **Shift** is the usual A-to-B delay. **Target jitter** is how much earlier one B arrives.

Run all cells. Play each trial, then open its answer. Start at a comfortable low volume.

In [ ]:
#@title Load
import importlib.util
import subprocess
import sys
from pathlib import Path

REF = "c5cb7c9a02b9d9a803fb7e7298166e461b6daba3"
URL = "https://github.com/MeysamAmirsardari/SeqSFG_task.git"
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    REPO = Path("/content/seqsfg_tpair_" + REF[:7])
    if not REPO.exists():
        subprocess.run(["git", "clone", "--quiet", URL, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REF], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "tpair" / "stimulus.py").exists()), None)
    if REPO is None:
        raise RuntimeError("Open in Colab, or run from the SeqSFG repository.")
for name in ["numpy", "scipy", "matplotlib"]:
    if importlib.util.find_spec(name) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", name], check=True)
sys.path.insert(0, str(REPO))

import csv
import io
import json
import numpy as np
import matplotlib.pyplot as plt
get_ipython().run_line_magic("matplotlib", "inline")
from matplotlib.patches import Rectangle
from IPython.display import Audio, HTML, display
from tpair.config import Config, conditions
from tpair.stimulus import build_trial, render_trial
from tpair.analysis import thresholds, replay_tracks

plt.rcParams.update({"font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 120})

In [ ]:
#@title Session data
CONFIG = {'sample_rate': 48000, 'monaural': False, 'tone_amplitude': 0.05, 'tone_level_db_spl': 65.0, 'level_rove_db': 3.0, 'f_a_hz': 700.0, 'f_b_hz': 2262.0, 'tone_ms': 75.0, 'ramp_ms': 5.0, 'a_leads': True, 'rate_hz': 3.0, 'n_pairs': 6, 'pair_jitter_ms': 35.0, 'interval_ms': 2000.0, 'lead_ms': 50.0, 'isi_ms': 500.0, 'lags_ms': [0.0, 18.75, 37.5, 56.25, 75.0], 'include_b_alone': True, 'target_from_end': [1, 2], 'delta_direction': 'backward', 'delta_start_ms': 10.0, 'delta_min_ms': 0.25, 'delta_max_ms': 70.0, 'delta_max_b_alone_ms': 120.0, 'n_down': 3, 'step_factors': [4.0, 2.0, 1.4142135623730951], 'reversals_per_step': [1, 2], 'n_final_reversals': 6, 'max_trials_per_track': 90, 'tracks_per_condition': 1, 'catch_rate': 0.07, 'catch_delta_ms': 60.0, 'catch_at_lag_ms': 0.0, 'max_catch_miss_rate': 0.15, 'feedback': True, 'practice_trials': 12, 'practice_criterion': 0.75, 'practice_delta_ms': 60.0, 'familiarise': True, 'response_timeout_s': 6.0, 'break_every': 1, 'break_every_trials': 70, 'max_same_condition_run': 2, 'tracks_per_block': None, 'n_boot': 4000, 'lapse_max': 0.1}
TRIAL_CSV = 'phase,condition,kind,track_id,track_trial_index,delta_ms,delta_realised_ms,correct,timed_out\npractice,lag_0,pair,,,60.000000,-60.000000,1,0\npractice,lag_0,pair,,,60.000000,-60.000000,1,0\npractice,lag_0,pair,,,60.000000,-60.000000,1,0\npractice,lag_0,pair,,,60.000000,-60.000000,1,0\npractice,lag_0,pair,,,60.000000,-60.000000,1,0\npractice,lag_0,pair,,,60.000000,-60.000000,1,0\npractice,lag_0,pair,,,60.000000,-60.000000,1,0\npractice,lag_0,pair,,,60.000000,-60.000000,1,0\npractice,lag_0,pair,,,60.000000,-60.000000,1,0\npractice,lag_0,pair,,,60.000000,-60.000000,1,0\npractice,lag_0,pair,,,60.000000,-60.000000,1,0\npractice,lag_0,pair,,,60.000000,-60.000000,1,0\nmain,lag_75,pair,0,0,10.000000,-10.000000,0,0\nmain,lag_56p25,pair,1,0,10.000000,-10.000000,1,0\nmain,lag_37p5,pair,2,0,10.000000,-10.000000,1,0\nmain,lag_18p75,pair,3,0,10.000000,-10.000000,1,0\nmain,b_alone,b_alone,4,0,10.000000,-10.000000,0,0\nmain,lag_0,pair,5,0,10.000000,-10.000000,1,0\nmain,lag_75,pair,0,1,40.000000,-40.000000,1,0\nmain,lag_56p25,pair,1,1,10.000000,-10.000000,1,0\nmain,lag_37p5,pair,2,1,10.000000,-10.000000,0,0\nmain,lag_18p75,pair,3,1,10.000000,-10.000000,1,0\nmain,b_alone,b_alone,4,1,40.000000,-40.000000,0,0\nmain,lag_0,pair,5,1,10.000000,-10.000000,1,0\nmain,lag_75,pair,0,2,40.000000,-40.000000,1,0\nmain,lag_56p25,pair,1,2,10.000000,-10.000000,1,0\nmain,lag_37p5,pair,2,2,40.000000,-40.000000,1,0\nmain,lag_18p75,pair,3,2,10.000000,-10.000000,0,0\nmain,b_alone,b_alone,4,2,120.000000,-120.000000,1,0\nmain,lag_0,pair,5,2,10.000000,-10.000000,1,0\nmain,lag_75,pair,0,3,40.000000,-40.000000,1,0\nmain,lag_56p25,pair,1,3,2.500000,-2.500000,1,0\nmain,lag_37p5,pair,2,3,40.000000,-40.000000,1,0\nmain,lag_18p75,pair,3,3,40.000000,-40.000000,1,0\nmain,b_alone,b_alone,4,3,120.000000,-120.000000,0,0\nmain,lag_0,pair,5,3,2.500000,-2.500000,0,0\ncatch,lag_0,pair,0,,60.000000,-60.000000,1,0\nmain,lag_56p25,pair,1,4,2.500000,-2.500000,0,0\nmain,lag_37p5,pair,2,4,40.000000,-40.000000,1,0\nmain,lag_18p75,pair,3,4,40.000000,-40.000000,1,0\ncatch,lag_0,pair,4,,60.000000,-60.000000,1,0\nmain,lag_0,pair,5,4,5.000000,-5.000000,1,0\nmain,lag_75,pair,0,4,20.000000,-20.000000,1,0\nmain,lag_56p25,pair,1,5,5.000000,-5.000000,1,0\nmain,lag_37p5,pair,2,5,20.000000,-20.000000,1,0\nmain,lag_18p75,pair,3,5,40.000000,-40.000000,0,0\nmain,b_alone,b_alone,4,4,120.000000,-120.000000,1,0\nmain,lag_0,pair,5,5,5.000000,-5.000000,1,0\ncatch,lag_0,pair,0,,60.000000,-60.000000,1,0\nmain,lag_56p25,pair,1,6,5.000000,-5.000000,1,0\nmain,lag_37p5,pair,2,6,20.000000,-20.000000,0,0\nmain,lag_18p75,pair,3,6,70.000000,-70.000000,1,0\nmain,b_alone,b_alone,4,5,120.000000,-120.000000,1,0\nmain,lag_0,pair,5,6,5.000000,-5.000000,1,0\nmain,lag_75,pair,0,5,20.000000,-20.000000,0,0\nmain,lag_56p25,pair,1,7,5.000000,-5.000000,1,0\nmain,lag_37p5,pair,2,7,40.000000,-40.000000,1,0\nmain,lag_18p75,pair,3,7,70.000000,-70.000000,1,0\nmain,b_alone,b_alone,4,6,120.000000,-120.000000,1,0\nmain,lag_0,pair,5,7,2.500000,-2.500000,1,0\nmain,lag_75,pair,0,6,40.000000,-40.000000,1,0\nmain,lag_56p25,pair,1,8,2.500000,-2.500000,1,0\nmain,lag_37p5,pair,2,8,40.000000,-40.000000,1,0\nmain,lag_18p75,pair,3,8,70.000000,-70.000000,1,0\nmain,b_alone,b_alone,4,7,60.000000,-60.000000,1,0\nmain,lag_0,pair,5,8,2.500000,-2.500000,1,0\nmain,lag_75,pair,0,7,40.000000,-40.000000,1,0\nmain,lag_56p25,pair,1,9,2.500000,-2.500000,1,0\nmain,lag_37p5,pair,2,9,40.000000,-40.000000,1,0\nmain,lag_18p75,pair,3,9,35.000000,-35.000000,1,0\nmain,b_alone,b_alone,4,8,60.000000,-60.000000,1,0\nmain,lag_0,pair,5,9,2.500000,-2.500000,1,0\nmain,lag_75,pair,0,8,40.000000,-40.000000,1,0\nmain,lag_56p25,pair,1,10,2.500000,-2.500000,1,0\ncatch,lag_0,pair,2,,60.000000,-60.000000,1,0\nmain,lag_18p75,pair,3,10,35.000000,-35.000000,1,0\nmain,b_alone,b_alone,4,9,60.000000,-60.000000,1,0\nmain,lag_0,pair,5,10,1.250000,-1.250000,1,0\nmain,lag_75,pair,0,9,28.284271,-28.291667,1,0\nmain,lag_56p25,pair,1,11,1.250000,-1.250000,1,0\nmain,lag_37p5,pair,2,10,28.284271,-28.291667,1,0\nmain,lag_18p75,pair,3,11,35.000000,-35.000000,1,0\nmain,b_alone,b_alone,4,10,30.000000,-30.000000,0,0\nmain,lag_0,pair,5,11,1.250000,-1.250000,1,0\nmain,lag_75,pair,0,10,28.284271,-28.270833,0,0\ncatch,lag_0,pair,1,,60.000000,-60.000000,1,0\nmain,lag_37p5,pair,2,11,28.284271,-28.291667,1,0\nmain,lag_18p75,pair,3,12,17.500000,-17.500000,1,0\nmain,b_alone,b_alone,4,11,60.000000,-60.000000,1,0\nmain,lag_0,pair,5,12,1.250000,-1.250000,1,0\ncatch,lag_0,pair,0,,60.000000,-60.000000,1,0\nmain,lag_56p25,pair,1,12,1.250000,-1.250000,1,0\nmain,lag_37p5,pair,2,12,28.284271,-28.291667,1,0\nmain,lag_18p75,pair,3,13,17.500000,-17.500000,0,0\nmain,b_alone,b_alone,4,12,60.000000,-60.000000,1,0\nmain,lag_0,pair,5,13,0.625000,-0.625000,0,0\nmain,lag_75,pair,0,11,40.000000,-40.000000,1,0\nmain,lag_56p25,pair,1,13,1.250000,-1.250000,0,0\nmain,lag_37p5,pair,2,13,20.000000,-20.000000,0,0\nmain,lag_18p75,pair,3,14,35.000000,-35.000000,1,0\ncatch,lag_0,pair,4,,60.000000,-60.000000,1,0\nmain,lag_0,pair,5,14,0.883883,-0.875000,1,0\nmain,lag_75,pair,0,12,40.000000,-40.000000,1,0\nmain,lag_56p25,pair,1,14,1.767767,-1.770833,1,0\ncatch,lag_0,pair,2,,60.000000,-60.000000,1,0\nmain,lag_18p75,pair,3,15,35.000000,-35.000000,1,0\nmain,b_alone,b_alone,4,13,60.000000,-60.000000,1,0\nmain,lag_0,pair,5,15,0.883883,-0.875000,0,0\nmain,lag_75,pair,0,13,40.000000,-40.000000,1,0\nmain,lag_56p25,pair,1,15,1.767767,-1.770833,1,0\nmain,lag_37p5,pair,2,14,28.284271,-28.291667,1,0\nmain,lag_18p75,pair,3,16,35.000000,-35.000000,1,0\nmain,b_alone,b_alone,4,14,42.426407,-42.416667,1,0\nmain,lag_0,pair,5,16,1.250000,-1.250000,1,0\nmain,lag_75,pair,0,14,28.284271,-28.291667,1,0\nmain,lag_56p25,pair,1,16,1.767767,-1.770833,0,0\nmain,lag_37p5,pair,2,15,28.284271,-28.291667,1,0\nmain,lag_18p75,pair,3,17,24.748737,-24.750000,1,0\nmain,b_alone,b_alone,4,15,42.426407,-42.416667,1,0\nmain,lag_0,pair,5,17,1.250000,-1.250000,0,0\nmain,lag_75,pair,0,15,28.284271,-28.291667,1,0\nmain,lag_56p25,pair,1,17,2.500000,-2.500000,1,0\nmain,lag_37p5,pair,2,16,28.284271,-28.291667,1,0\nmain,lag_18p75,pair,3,18,24.748737,-24.750000,1,0\nmain,b_alone,b_alone,4,16,42.426407,-42.437500,1,0\nmain,lag_0,pair,5,18,1.767767,-1.770833,1,0\nmain,lag_75,pair,0,16,28.284271,-28.291667,1,0\nmain,lag_56p25,pair,1,18,2.500000,-2.500000,1,0\nmain,lag_37p5,pair,2,17,20.000000,-20.000000,1,0\nmain,lag_18p75,pair,3,19,24.748737,-24.750000,1,0\nmain,b_alone,b_alone,4,17,30.000000,-30.000000,1,0\nmain,lag_0,pair,5,19,1.767767,-1.750000,1,0\nmain,lag_75,pair,0,17,20.000000,-20.000000,1,0\ncatch,lag_0,pair,1,,60.000000,-60.000000,1,0\ncatch,lag_0,pair,2,,60.000000,-60.000000,1,0\ncatch,lag_0,pair,3,,60.000000,-60.000000,1,0\nmain,b_alone,b_alone,4,18,30.000000,-30.000000,1,0\nmain,lag_0,pair,5,20,1.767767,-1.770833,1,0\nmain,lag_75,pair,0,18,20.000000,-20.000000,1,0\nmain,lag_56p25,pair,1,19,2.500000,-2.500000,1,0\nmain,lag_37p5,pair,2,18,20.000000,-20.000000,1,0\nmain,lag_18p75,pair,3,20,17.500000,-17.500000,1,0\nmain,b_alone,b_alone,4,19,30.000000,-30.000000,1,0\nmain,lag_0,pair,5,21,1.250000,-1.250000,0,0\nmain,lag_75,pair,0,19,20.000000,-20.000000,1,0\nmain,lag_56p25,pair,1,20,1.767767,-1.750000,0,0\nmain,lag_37p5,pair,2,19,20.000000,-20.000000,1,0\nmain,lag_18p75,pair,3,21,17.500000,-17.500000,1,0\nmain,b_alone,b_alone,4,20,21.213203,-21.208333,1,0\nmain,lag_0,pair,5,22,1.767767,-1.770833,1,0\nmain,lag_75,pair,0,20,14.142136,-14.145833,1,0\nmain,lag_56p25,pair,1,21,2.500000,-2.500000,0,0\nmain,lag_37p5,pair,2,20,14.142136,-14.145833,0,0\nmain,lag_18p75,pair,3,22,17.500000,-17.500000,0,0\nmain,b_alone,b_alone,4,21,21.213203,-21.208333,0,0\nmain,lag_0,pair,5,23,1.767767,-1.770833,1,0\nmain,lag_75,pair,0,21,14.142136,-14.145833,0,0\nmain,lag_56p25,pair,1,22,3.535534,-3.520833,0,0\nmain,lag_37p5,pair,2,21,20.000000,-20.000000,1,0\nmain,lag_18p75,pair,3,23,24.748737,-24.750000,1,0\nmain,b_alone,b_alone,4,22,30.000000,-30.000000,0,0\nmain,lag_0,pair,5,24,1.767767,-1.770833,0,0\nmain,lag_75,pair,0,22,20.000000,-20.000000,1,0\nmain,lag_56p25,pair,1,23,5.000000,-5.000000,1,0\nmain,lag_37p5,pair,2,22,20.000000,-20.000000,0,0\nmain,lag_18p75,pair,3,24,24.748737,-24.750000,1,0\nmain,b_alone,b_alone,4,23,42.426407,-42.416667,1,0\nmain,lag_0,pair,5,25,2.500000,-2.500000,1,0\nmain,lag_75,pair,0,23,20.000000,-20.000000,1,0\nmain,lag_56p25,pair,1,24,5.000000,-5.000000,1,0\nmain,lag_37p5,pair,2,23,28.284271,-28.270833,0,0\nmain,lag_18p75,pair,3,25,24.748737,-24.750000,1,0\nmain,b_alone,b_alone,4,24,42.426407,-42.416667,1,0\nmain,lag_0,pair,5,26,2.500000,-2.500000,1,0\nmain,lag_75,pair,0,24,20.000000,-20.000000,0,0\nmain,lag_56p25,pair,1,25,5.000000,-5.000000,0,0\nmain,lag_37p5,pair,2,24,40.000000,-40.000000,1,0\nmain,lag_18p75,pair,3,26,17.500000,-17.500000,1,0\nmain,b_alone,b_alone,4,25,42.426407,-42.416667,1,0\nmain,lag_0,pair,5,27,2.500000,-2.500000,1,0\nmain,lag_75,pair,0,25,28.284271,-28.270833,1,0\nmain,lag_56p25,pair,1,26,7.071068,-7.062500,1,0\nmain,lag_37p5,pair,2,25,40.000000,-40.000000,0,0\nmain,lag_18p75,pair,3,27,17.500000,-17.500000,0,0\nmain,b_alone,b_alone,4,26,30.000000,-30.000000,0,0\nmain,lag_0,pair,5,28,1.767767,-1.770833,1,0\nmain,lag_75,pair,0,26,28.284271,-28.291667,1,0\nmain,lag_56p25,pair,1,27,7.071068,-7.062500,1,0\nmain,lag_37p5,pair,2,26,56.568542,-56.583333,1,0\nmain,lag_18p75,pair,3,28,24.748737,-24.750000,1,0\nmain,b_alone,b_alone,4,27,42.426407,-42.416667,1,0\nmain,lag_0,pair,5,29,1.767767,-1.770833,1,0\nmain,lag_75,pair,0,27,28.284271,-28.291667,0,0\nmain,lag_56p25,pair,1,28,7.071068,-7.062500,0,0\nmain,lag_37p5,pair,2,27,56.568542,-56.562500,1,0\nmain,lag_18p75,pair,3,29,24.748737,-24.750000,1,0\nmain,b_alone,b_alone,4,28,42.426407,-42.416667,1,0\nmain,lag_0,pair,5,30,1.767767,-1.770833,0,0\nmain,lag_75,pair,0,28,40.000000,-40.000000,1,0\nmain,lag_56p25,pair,1,29,10.000000,-10.000000,0,0\nmain,lag_37p5,pair,2,28,56.568542,-56.562500,1,0\ncatch,lag_0,pair,3,,60.000000,-60.000000,1,0\nmain,b_alone,b_alone,4,29,42.426407,-42.416667,1,0\nmain,lag_0,pair,5,31,2.500000,-2.500000,1,0\nmain,lag_75,pair,0,29,40.000000,-40.000000,1,0\nmain,lag_56p25,pair,1,30,14.142136,-14.145833,1,0\nmain,lag_37p5,pair,2,29,40.000000,-40.000000,1,0\nmain,lag_18p75,pair,3,30,24.748737,-24.750000,1,0\nmain,b_alone,b_alone,4,30,30.000000,-30.000000,0,0\nmain,lag_0,pair,5,32,2.500000,-2.500000,1,0\nmain,lag_75,pair,0,30,40.000000,-40.000000,1,0\nmain,lag_56p25,pair,1,31,14.142136,-14.145833,1,0\nmain,lag_37p5,pair,2,30,40.000000,-40.000000,1,0\nmain,lag_18p75,pair,3,31,17.500000,-17.500000,1,0\nmain,b_alone,b_alone,4,31,42.426407,-42.416667,0,0\nmain,lag_0,pair,5,33,2.500000,-2.500000,0,0\nmain,lag_75,pair,0,31,28.284271,-28.291667,1,0\nmain,lag_56p25,pair,1,32,14.142136,-14.125000,0,0\nmain,lag_37p5,pair,2,31,40.000000,-40.000000,1,0\nmain,lag_18p75,pair,3,32,17.500000,-17.500000,1,0\nmain,b_alone,b_alone,4,32,60.000000,-60.000000,0,0\nmain,lag_0,pair,5,34,3.535534,-3.541667,1,0\nmain,lag_75,pair,0,32,28.284271,-28.291667,1,0\nmain,lag_56p25,pair,1,33,20.000000,-20.000000,0,0\nmain,lag_37p5,pair,2,32,28.284271,-28.270833,0,0\nmain,lag_18p75,pair,3,33,17.500000,-17.500000,1,0\nmain,b_alone,b_alone,4,33,84.852814,-84.854167,1,0\ncatch,lag_0,pair,5,,60.000000,-60.000000,1,0\nmain,lag_75,pair,0,33,28.284271,-28.291667,1,0\nmain,lag_56p25,pair,1,34,28.284271,-28.291667,1,0\nmain,lag_37p5,pair,2,33,40.000000,-40.000000,1,0\nmain,lag_18p75,pair,3,34,12.374369,-12.375000,0,0\nmain,b_alone,b_alone,4,34,84.852814,-84.854167,1,0\nmain,lag_0,pair,5,35,3.535534,-3.541667,1,0\nmain,lag_75,pair,0,34,20.000000,-20.000000,1,0\nmain,lag_56p25,pair,1,35,28.284271,-28.270833,1,0\nmain,lag_37p5,pair,2,34,40.000000,-40.000000,1,0\nmain,lag_18p75,pair,3,35,17.500000,-17.500000,0,0\nmain,b_alone,b_alone,4,35,84.852814,-84.854167,1,0\nmain,lag_0,pair,5,36,3.535534,-3.520833,1,0\nmain,lag_75,pair,0,35,20.000000,-20.000000,1,0\ncatch,lag_0,pair,1,,60.000000,-60.000000,1,0\nmain,lag_37p5,pair,2,35,40.000000,-40.000000,1,0\nmain,lag_18p75,pair,3,36,24.748737,-24.750000,1,0\nmain,lag_0,pair,5,37,2.500000,-2.500000,1,0\nmain,lag_75,pair,0,36,20.000000,-20.000000,1,0\nmain,lag_56p25,pair,1,36,28.284271,-28.291667,1,0\nmain,lag_18p75,pair,3,37,24.748737,-24.750000,0,0\nmain,lag_0,pair,5,38,2.500000,-2.500000,1,0\nmain,lag_75,pair,0,37,14.142136,-14.145833,1,0\ncatch,lag_0,pair,1,,60.000000,-60.000000,1,0\nmain,lag_18p75,pair,3,38,35.000000,-35.000000,1,0\nmain,lag_0,pair,5,39,2.500000,-2.500000,1,0\nmain,lag_75,pair,0,38,14.142136,-14.145833,0,0\nmain,lag_56p25,pair,1,37,20.000000,-20.000000,1,0\nmain,lag_18p75,pair,3,39,35.000000,-35.000000,1,0\nmain,lag_0,pair,5,40,1.767767,-1.770833,1,0\nmain,lag_75,pair,0,39,20.000000,-20.000000,1,0\nmain,lag_56p25,pair,1,38,20.000000,-20.000000,1,0\nmain,lag_18p75,pair,3,40,35.000000,-35.000000,1,0\ncatch,lag_0,pair,5,,60.000000,-60.000000,0,0\nmain,lag_75,pair,0,40,20.000000,-20.000000,0,0\nmain,lag_56p25,pair,1,39,20.000000,-20.000000,0,0\nmain,lag_0,pair,5,41,1.767767,-1.770833,1,0\nmain,lag_75,pair,0,41,28.284271,-28.291667,1,0\nmain,lag_56p25,pair,1,40,28.284271,-28.270833,1,0\nmain,lag_0,pair,5,42,1.767767,-1.770833,1,0\nmain,lag_75,pair,0,42,28.284271,-28.270833,1,0\nmain,lag_56p25,pair,1,41,28.284271,-28.291667,1,0\nmain,lag_0,pair,5,43,1.250000,-1.250000,1,0\nmain,lag_75,pair,0,43,28.284271,-28.291667,1,0\nmain,lag_56p25,pair,1,42,28.284271,-28.291667,1,0\nmain,lag_0,pair,5,44,1.250000,-1.250000,0,0\nmain,lag_56p25,pair,1,43,20.000000,-20.000000,0,0\n'
cfg = Config.from_dict(CONFIG)
rows = list(csv.DictReader(io.StringIO(TRIAL_CSV)))
conds = {c.name: c for c in conditions(cfg)}
JITTERS = [2, 5, 10, 20, 40, 60]

In [ ]:
#@title Plot and audio helpers
A_COLOR, B_COLOR = "#727c86", "#286da8"

def timing_plot(trial):
    fig, axes = plt.subplots(1, 2, figsize=(12, 2.4), sharey=True)
    for number, (ax, iv) in enumerate(zip(axes, [trial.first, trial.second]), 1):
        for onsets, y, color in [(iv.a_onsets_ms, 0, A_COLOR),
                                 (iv.b_onsets_ms, 1, B_COLOR)]:
            for onset in onsets:
                if np.isfinite(onset):
                    ax.add_patch(Rectangle((onset / 1000, y - .16),
                                           cfg.tone_ms / 1000, .32, color=color))
        if iv.target_pair is not None:
            k = iv.target_pair
            original = (iv.b_onsets_ms[k] - iv.delta_ms) / 1000
            moved = iv.b_onsets_ms[k] / 1000
            ax.add_patch(Rectangle((original, .80), cfg.tone_ms / 1000, .40,
                                  fill=False, ec="#c86a24", ls=":"))
            ax.annotate("", (moved, 1.34), (original, 1.34),
                        arrowprops={"arrowstyle": "->", "color": "#c86a24"})
        ax.set(xlim=(0, 2), ylim=(-.4, 1.6), xlabel="Time (s)", title=f"Sound {number}")
        ax.set_yticks([0, 1], ["A · 700 Hz", "B · 2262 Hz"])
    fig.suptitle("Example: B arrives 20 ms early in sound 2", fontsize=11)
    fig.tight_layout()
    plt.show()

def block(name, seed):
    cond = conds[name]
    # A fixed seed holds the nuisance timing and target location across this demo ladder.
    # The two sounds within each trial still have independently drawn token jitter.
    demo = build_trial(cfg, cond, 20, np.random.default_rng(seed), target_position=2)
    timing_plot(demo)
    peak = 0.0
    for delta in JITTERS:
        tr = build_trial(cfg, cond, delta, np.random.default_rng(seed))
        waveform = render_trial(cfg, tr)
        peak = max(peak, float(np.max(np.abs(waveform))))
        assert waveform.size == round(4.5 * cfg.sample_rate)
        assert np.isfinite(waveform).all() and np.max(np.abs(waveform)) < 1
        display(HTML(f"<b>Target jitter: {delta} ms</b>"))
        display(Audio(waveform, rate=cfg.sample_rate, normalize=False, autoplay=False))
        place = "last" if tr.target_pair == cfg.n_pairs - 1 else "second-to-last"
        display(HTML(f"<details><summary>Answer</summary>Sound {tr.target_position}; "
                     f"the {place} B arrives {delta} ms early.</details>"))
    return peak

## Listen

The examples are grouped by shift, with target jitter increasing within each block. These are listening examples, not a threshold test. The recorded session interleaved conditions.

### Shift 0% · 0 ms

In [ ]:
#@title Plot and trials
peak_0 = block('lag_0', 101)

### Shift 25% · 18.75 ms

In [ ]:
#@title Plot and trials
peak_25 = block('lag_18p75', 102)

### Shift 50% · 37.5 ms

In [ ]:
#@title Plot and trials
peak_50 = block('lag_37p5', 103)

### Shift 75% · 56.25 ms

In [ ]:
#@title Plot and trials
peak_75 = block('lag_56p25', 104)

### Shift 100% · 75 ms

In [ ]:
#@title Plot and trials
peak_100 = block('lag_75', 105)

### B alone

In [ ]:
#@title Plot and trials
peak_control = block('b_alone', 106)

## L01 · Session 02 · 19 October 2026

246 main trials. Catch: 15/16. One track per condition.

75% is omitted from this plot. Its recorded data are retained. The line joins staircase thresholds; the dots also show the fit and percent correct.

In [ ]:
#@title Results
results = thresholds(rows, cfg)
audits = replay_tracks(rows, cfg)
assert len(audits) == 6 and all(a["logged_deltas_match"] for a in audits.values())
assert all(a["converged"] and not a["ceiling_inside_threshold"]
           and not a["floor_inside_threshold"] for a in audits.values())
shown = [results[c] for c in ["lag_0", "lag_18p75", "lag_37p5", "lag_75"]]
x = np.array([r.lag_ms for r in shown])
y = np.array([r.geomean_ms for r in shown])
fit = np.array([r.fit.threshold_ms for r in shown])
assert np.allclose(y, [1.8728838461, 20.8111245125, 28.2842712475, 25.1984209979])
assert all(r.lag_ms != 56.25 for r in shown)

fig, (ax, rate_ax) = plt.subplots(1, 2, figsize=(11, 4.3),
                                gridspec_kw={"width_ratios": [1.6, 1]})
ax.plot(x, y, "o-", color=B_COLOR, lw=1.8, ms=6, label="Staircase")
ax.scatter(x, fit, marker="s", color="#c86a24", s=32, label="Psychometric fit", zorder=4)
solo = results["b_alone"]
ax.axhline(solo.geomean_ms, color=A_COLOR, ls="--", lw=1.2,
           label=f"B alone: {solo.geomean_ms:.1f} ms")
for xx, yy in zip(x, y):
    ax.annotate(f"{yy:.1f}", (xx, yy), xytext=(0, 9), textcoords="offset points",
                ha="center", color=B_COLOR, fontsize=9)
ax.set_yscale("log")
ax.set_ylim(1, 75)
ax.set_yticks([1, 2, 5, 10, 20, 40, 60], ["1", "2", "5", "10", "20", "40", "60"])
ax.set_xticks(x, ["0%", "25%", "50%", "100%"])
ax.set(xlabel="A-to-B shift", ylabel="Displacement threshold (ms)", title="Timing sensitivity")
ax.legend(loc="upper left", fontsize=8, framealpha=1, edgecolor="none")
ax.grid(axis="y", alpha=.15)
rate_rows = shown + [solo]
rate_ax.scatter(range(5), [100*r.prop_correct for r in rate_rows], color=B_COLOR, s=42)
rate_ax.axhline(50, color=A_COLOR, ls=":", lw=1)
rate_ax.set_xticks(range(5), ["0%", "25%", "50%", "100%", "B alone"])
rate_ax.set(xlim=(-.5, 4.5), ylim=(45, 100), ylabel="Correct (%)", title="Main-trial responses")
rate_ax.grid(axis="y", alpha=.15)
fig.tight_layout()
plt.show()

Lower thresholds mean better timing sensitivity. Response rates come from adaptive trials with different difficulties. This session shows a large synchrony advantage; it does not establish a binding limit.